# Q-Shield XAI: Grad-CAM + SHAP Analysis (Self-Sufficient)

This notebook runs even if the v3 checkpoint is missing — it will train v3 inline as fallback.

**Generates paper figures:**
- Grad-CAM heatmaps (individual samples + class average)
- Embedding space distance distributions
- SHAP importance on embedding dimensions

**Author:** Nicolas A. Llerena Silva (UTEC)

In [ ]:
# ============================================================
# 0. SETUP — Mount Drive + Find BASE
# ============================================================
import sys, os, glob

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Try multiple BASE paths
CANDIDATES = [
    '/content/drive/MyDrive/Proyecto_Quishing_Detection_Nicolas',
    '/content/drive/MyDrive/QShield',
    '.',
]
BASE = None
for c in CANDIDATES:
    if os.path.exists(c):
        BASE = c
        break
assert BASE is not None, 'No valid BASE directory found'
print(f'BASE = {BASE}')

# List contents
print('\nFiles in BASE:')
for f in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, f)
    if os.path.isfile(full):
        size = os.path.getsize(full) / 1024 / 1024
        print(f'  {f:<45} {size:>8.1f} MB')
    else:
        print(f'  {f}/   [DIR]')

In [ ]:
# ============================================================
# 0.1 GPU CHECK & INSTALL DEPS
# ============================================================
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm shap

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist
from PIL import Image
import pickle, zipfile, random, time, copy, json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram/1e9:.1f} GB')

---
## 1. MODEL ARCHITECTURE (v3)

In [ ]:
# ============================================================
# 1. MODEL CLASSES (must match v3 exactly)
# ============================================================

class MobileNetV2Embedding(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        mn = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        orig = mn.features[0][0]
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(orig.weight.mean(dim=1, keepdim=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(512, emb_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return F.normalize(self.projection(x), p=2, dim=1)


class SiameseQRNet(nn.Module):
    def __init__(self, emb_dim=128, pretrained=True, dropout=0.35):
        super().__init__()
        self.backbone = MobileNetV2Embedding(emb_dim, pretrained, dropout)
    def forward_one(self, x): return self.backbone(x)
    def forward(self, x1, x2): return self.backbone(x1), self.backbone(x2)


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.5):
        super().__init__(); self.margin = margin
    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        return ((1-y)*0.5*d.pow(2) + y*0.5*F.relu(self.margin-d).pow(2)).mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__(); self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        pt = p * targets + (1-p) * (1-targets)
        at = self.alpha * targets + (1-self.alpha) * (1-targets)
        return (at * (1-pt).pow(self.gamma) * bce).mean()


class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )
    def set_backbone_grad(self, rg):
        for p in self.backbone.parameters(): p.requires_grad = rg
    def forward(self, x):
        return self.head(self.backbone(x))

print('Model classes defined.')

---
## 2. CHECKPOINT LOADING (with inline training fallback)

In [ ]:
# ============================================================
# 2.1 FIND OR TRAIN CHECKPOINT
# ============================================================

# Look for any existing classifier checkpoint
CLASSIFIER_CANDIDATES = [
    'classifier_v3_phase2.pth',
    'classifier_phase2.pth',
    'classifier_v2_phase2.pth',
    'classifier_phase_2.pth',
]

ckpt_path = None
for name in CLASSIFIER_CANDIDATES:
    p = os.path.join(BASE, name)
    if os.path.exists(p):
        ckpt_path = p
        break

if ckpt_path is None:
    matches = glob.glob(os.path.join(BASE, 'classifier*.pth'))
    if matches:
        ckpt_path = matches[0]

HAS_CKPT = ckpt_path is not None
print(f'Existing checkpoint found: {HAS_CKPT}')
if HAS_CKPT:
    print(f'  Path: {ckpt_path}')
else:
    print('  No checkpoint in Drive. Will train v3 inline (adds ~40 min to notebook).')

In [ ]:
# ============================================================
# 2.2 LOAD DATA (Trad is small, always needed)
# ============================================================
WORK = '/content/qshield_data'
os.makedirs(WORK, exist_ok=True)

trad_dir = os.path.join(WORK, 'trad')
trad_zip = os.path.join(BASE, 'QuishingDataset.zip')
assert os.path.exists(trad_zip), f'QuishingDataset.zip not found in {BASE}'

if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(trad_zip) as z:
        z.extractall(trad_dir)

with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)
print(f'Trad loaded: {trad_qr.shape}')

# Splits
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=SEED)
qr_tr, lab_tr = trad_qr[idx_tr], trad_labels[idx_tr]
qr_val, lab_val = trad_qr[idx_val], trad_labels[idx_val]
print(f'Train: {len(qr_tr)}  Val: {len(qr_val)}')

In [ ]:
# ============================================================
# 2.3 LOAD CIC (only if training inline)
# ============================================================
HAS_CIC = False
cic_b_tr = cic_m_tr = cic_b_val = cic_m_val = None

if not HAS_CKPT:
    cic_b_dir = os.path.join(WORK, 'cic_benign')
    cic_m_dir = os.path.join(WORK, 'cic_malicious')
    zip_b = os.path.join(BASE, 'QR_benign_430K.zip')
    zip_m = os.path.join(BASE, 'QR_malicious_576K.zip')

    if os.path.exists(zip_b) and os.path.exists(zip_m):
        if not os.path.exists(cic_b_dir) or len(os.listdir(cic_b_dir)) == 0:
            os.makedirs(cic_b_dir, exist_ok=True)
            print('Extracting CIC benign...')
            with zipfile.ZipFile(zip_b) as z: z.extractall(cic_b_dir)
        if not os.path.exists(cic_m_dir) or len(os.listdir(cic_m_dir)) == 0:
            os.makedirs(cic_m_dir, exist_ok=True)
            print('Extracting CIC malicious...')
            with zipfile.ZipFile(zip_m) as z: z.extractall(cic_m_dir)

        cic_b_files = sorted(glob.glob(os.path.join(cic_b_dir, '**', '*.png'), recursive=True))
        cic_m_files = sorted(glob.glob(os.path.join(cic_m_dir, '**', '*.png'), recursive=True))
        random.seed(SEED)
        cic_b_files = random.sample(cic_b_files, min(30000, len(cic_b_files)))
        cic_m_files = random.sample(cic_m_files, min(30000, len(cic_m_files)))

        sb = int(len(cic_b_files)*0.8); sm = int(len(cic_m_files)*0.8)
        cic_b_tr, cic_b_val = cic_b_files[:sb], cic_b_files[sb:]
        cic_m_tr, cic_m_val = cic_m_files[:sm], cic_m_files[sm:]
        HAS_CIC = True
        print(f'CIC: {len(cic_b_tr)+len(cic_m_tr)} train / {len(cic_b_val)+len(cic_m_val)} val')
    else:
        print('CIC zips not found. Will train on Trad only (still valid).')
else:
    print('Skipping CIC extraction (using existing checkpoint).')

In [ ]:
# ============================================================
# 2.4 INLINE TRAINING (only runs if no checkpoint)
# ============================================================
train_aug = T.Compose([T.RandomHorizontalFlip(p=0.5)])

class TradPairDataset(Dataset):
    def __init__(self, qr, labels, n, augment=False):
        self.qr = qr.astype(np.float32); self.labels = np.array(labels); self.n = n
        self.idx = {0: np.where(self.labels==0)[0], 1: np.where(self.labels==1)[0]}
        self.aug = train_aug if augment else None
    def __len__(self): return self.n
    def _tensor(self, i):
        t = torch.from_numpy(self.qr[i]).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        return self.aug(t) if self.aug else t
    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.choice(self.idx[c])
        c2 = c if same else 1-c; i2 = random.choice(self.idx[c2])
        return self._tensor(i1), self._tensor(i2), torch.tensor(0.0 if same else 1.0)

class CICPairDataset(Dataset):
    def __init__(self, b, m, n, augment=False):
        self.f = {0: b, 1: m}; self.n = n
        self.aug = train_aug if augment else None
    def __len__(self): return self.n
    def _load(self, c, i):
        img = Image.open(self.f[c][i]).convert('L').resize((224,224))
        t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0)
        return self.aug(t) if self.aug else t
    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0,1]); i1 = random.randint(0, len(self.f[c])-1)
        c2 = c if same else 1-c; i2 = random.randint(0, len(self.f[c2])-1)
        return self._load(c,i1), self._load(c2,i2), torch.tensor(0.0 if same else 1.0)

class ClassifyDataset(Dataset):
    def __init__(self, trad_qr=None, trad_labels=None, cic_b=None, cic_m=None, augment=False):
        self.items = []
        if trad_qr is not None:
            for i in range(len(trad_qr)): self.items.append(('trad', i, int(trad_labels[i])))
            self.trad_qr = trad_qr.astype(np.float32)
        if cic_b is not None:
            for i,_ in enumerate(cic_b): self.items.append(('cic_b', i, 0))
            for i,_ in enumerate(cic_m): self.items.append(('cic_m', i, 1))
            self.cic_b = cic_b; self.cic_m = cic_m
        self.aug = train_aug if augment else None
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        src, i, lbl = self.items[idx]
        if src == 'trad':
            t = torch.from_numpy(self.trad_qr[i]).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        else:
            files = self.cic_b if src == 'cic_b' else self.cic_m
            img = Image.open(files[i]).convert('L').resize((224,224))
            t = torch.from_numpy(np.array(img, dtype=np.float32)/255.0).unsqueeze(0)
        if self.aug: t = self.aug(t)
        return t, torch.tensor(float(lbl))


def train_v3_inline():
    """Train v3 if no checkpoint found. Compact version of notebook 06."""
    print('\n' + '='*60)
    print(' TRAINING v3 INLINE (checkpoint was not found)')
    print('='*60)
    BATCH = 128
    NUM_WORKERS = 4 if IN_COLAB else 0

    # Phase 1 data
    if HAS_CIC:
        p1_train = ConcatDataset([
            TradPairDataset(qr_tr, lab_tr, 25000, augment=True),
            CICPairDataset(cic_b_tr, cic_m_tr, 25000, augment=True),
        ])
        p1_val = ConcatDataset([
            TradPairDataset(qr_val, lab_val, 3000, augment=False),
            CICPairDataset(cic_b_val, cic_m_val, 3000, augment=False),
        ])
    else:
        p1_train = TradPairDataset(qr_tr, lab_tr, 25000, augment=True)
        p1_val = TradPairDataset(qr_val, lab_val, 3000, augment=False)
    p1_tl = DataLoader(p1_train, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    p1_vl = DataLoader(p1_val, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # Phase 1 training (reduced: 15 epochs for speed)
    model = SiameseQRNet(128, pretrained=True, dropout=0.35).to(device)
    criterion = ContrastiveLoss(margin=1.5)
    opt = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=2e-4)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=8, T_mult=1, eta_min=1e-6)
    best_vl = float('inf'); best_bb = None
    EPOCHS1 = 15

    print(f'\nPhase 1: {EPOCHS1} epochs')
    for ep in range(1, EPOCHS1+1):
        t0 = time.time()
        model.train()
        tl_sum = 0; n = 0
        for x1, x2, y in p1_tl:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            opt.zero_grad()
            e1, e2 = model(x1, x2)
            loss = criterion(e1, e2, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tl_sum += loss.item()*x1.size(0); n += x1.size(0)
        model.eval()
        vl = 0; nv = 0
        with torch.no_grad():
            for x1, x2, y in p1_vl:
                x1, x2, y = x1.to(device), x2.to(device), y.to(device)
                e1, e2 = model(x1, x2)
                vl += criterion(e1, e2, y).item()*x1.size(0); nv += x1.size(0)
        sched.step()
        tl_a, vl_a = tl_sum/n, vl/nv
        mk = ''
        if vl_a < best_vl:
            best_vl = vl_a; best_bb = copy.deepcopy(model.state_dict()); mk = ' *'
        print(f'  Ep {ep:>2}: tr={tl_a:.4f}  va={vl_a:.4f}  ({time.time()-t0:.0f}s){mk}')
    model.load_state_dict(best_bb)

    # Save Phase 1
    p1_path = os.path.join(BASE, 'siamese_v3_phase1.pth')
    torch.save(best_bb, p1_path)
    # Also local backup
    torch.save(best_bb, '/content/siamese_v3_phase1_local.pth')

    # Phase 2
    classifier = QRClassifier(model.backbone, emb_dim=128).to(device)
    p2_tr = ClassifyDataset(qr_tr, lab_tr, cic_b_tr if HAS_CIC else None, cic_m_tr if HAS_CIC else None, augment=True)
    p2_val = ClassifyDataset(qr_val, lab_val, cic_b_val if HAS_CIC else None, cic_m_val if HAS_CIC else None, augment=False)
    p2_tl = DataLoader(p2_tr, batch_size=256, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    p2_vl = DataLoader(p2_val, batch_size=512, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    focal = FocalLoss(alpha=0.5, gamma=2.0)
    # Frozen first
    classifier.set_backbone_grad(False)
    opt2 = optim.AdamW([p for p in classifier.parameters() if p.requires_grad], lr=5e-4, weight_decay=1e-4)
    best_auc = 0; best_cls = None
    EPOCHS2 = 12
    FROZEN = 3

    print(f'\nPhase 2: {EPOCHS2} epochs ({FROZEN} frozen + {EPOCHS2-FROZEN} unfrozen)')
    for ep in range(1, EPOCHS2+1):
        if ep == FROZEN+1:
            classifier.set_backbone_grad(True)
            opt2 = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=2e-4)
        classifier.train()
        tl_sum = 0; n = 0
        for imgs, lbls in p2_tl:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            opt2.zero_grad()
            loss = focal(classifier(imgs), lbls)
            loss.backward()
            opt2.step()
            tl_sum += loss.item()*imgs.size(0); n += imgs.size(0)
        classifier.eval()
        probs, true = [], []
        with torch.no_grad():
            for imgs, lbls in p2_vl:
                logits = classifier(imgs.to(device))
                probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
                true.extend(lbls.numpy().flatten())
        auc = roc_auc_score(true, probs)
        mk = ''
        if auc > best_auc:
            best_auc = auc; best_cls = copy.deepcopy(classifier.state_dict()); mk = ' *'
        print(f'  Ep {ep:>2} ({"frozen" if ep<=FROZEN else "unfrozen"}): AUC={auc:.4f}{mk}')
    classifier.load_state_dict(best_cls)

    # Save with VERIFICATION
    cls_path = os.path.join(BASE, 'classifier_v3_phase2.pth')
    torch.save(best_cls, cls_path)
    # Local backup
    local_backup = '/content/classifier_v3_phase2_local.pth'
    torch.save(best_cls, local_backup)

    # Verify Drive sync
    time.sleep(10)
    if os.path.exists(cls_path):
        print(f'\nVERIFIED: Drive checkpoint saved ({os.path.getsize(cls_path)/1024/1024:.1f} MB)')
    else:
        print(f'\nWARNING: Drive save failed. Using local: {local_backup}')
        cls_path = local_backup

    print(f'Best AUC: {best_auc:.4f}')
    return classifier, cls_path

# Run or skip training
if HAS_CKPT:
    print('Loading existing checkpoint...')
    classifier = QRClassifier(MobileNetV2Embedding(128, pretrained=False, dropout=0.35), emb_dim=128).to(device)
    state = torch.load(ckpt_path, map_location=device)
    classifier.load_state_dict(state)
    classifier.eval()
    print(f'Loaded from {ckpt_path}')
else:
    classifier, ckpt_path = train_v3_inline()

print(f'\nClassifier ready.')
print(f'Params: {sum(p.numel() for p in classifier.parameters()):,}')

---
## 3. GRAD-CAM — Visual Attention Maps

In [ ]:
# ============================================================
# 3.1 GRAD-CAM IMPLEMENTATION
# ============================================================

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor):
        self.model.eval()
        logit = self.model(input_tensor)
        self.model.zero_grad()
        logit.backward(torch.ones_like(logit))
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

target_layer = classifier.backbone.features[-1]
gradcam = GradCAM(classifier, target_layer)

def prepare_tensor(qr_array):
    t = torch.from_numpy(qr_array.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
    return t.to(device)

print('Grad-CAM ready.')

In [ ]:
# ============================================================
# 3.2 SAMPLE HEATMAPS (6 benign + 6 phishing)
# ============================================================
np.random.seed(42)
benign_idx = np.where(lab_val == 0)[0]
phish_idx = np.where(lab_val == 1)[0]
b_sample = np.random.choice(benign_idx, 6, replace=False)
p_sample = np.random.choice(phish_idx, 6, replace=False)

fig, axes = plt.subplots(4, 6, figsize=(18, 12))

for col, idx in enumerate(b_sample):
    qr = qr_val[idx]
    t = prepare_tensor(qr); t.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(t)
    with torch.no_grad():
        prob = torch.sigmoid(classifier(prepare_tensor(qr))).item()
    axes[0, col].imshow(qr, cmap='gray')
    axes[0, col].set_title(f'Benign #{idx}\nP(phish)={prob:.3f}', fontsize=9,
                            color='green' if prob < 0.5 else 'red')
    axes[0, col].axis('off')
    axes[1, col].imshow(qr, cmap='gray')
    axes[1, col].imshow(cam, cmap='jet', alpha=0.55)
    axes[1, col].axis('off')

for col, idx in enumerate(p_sample):
    qr = qr_val[idx]
    t = prepare_tensor(qr); t.requires_grad_(True)
    with torch.enable_grad():
        cam = gradcam(t)
    with torch.no_grad():
        prob = torch.sigmoid(classifier(prepare_tensor(qr))).item()
    axes[2, col].imshow(qr, cmap='gray')
    axes[2, col].set_title(f'Phishing #{idx}\nP(phish)={prob:.3f}', fontsize=9,
                            color='red' if prob > 0.5 else 'green')
    axes[2, col].axis('off')
    axes[3, col].imshow(qr, cmap='gray')
    axes[3, col].imshow(cam, cmap='jet', alpha=0.55)
    axes[3, col].axis('off')

fig.suptitle('Grad-CAM Attention — Q-Shield v3', fontweight='bold', fontsize=14, y=1.00)
plt.tight_layout()
save_path = os.path.join(BASE, 'fig_gradcam_samples.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_gradcam_samples.png', dpi=300, bbox_inches='tight')  # local backup
plt.show()
print(f'Saved: {save_path}')

In [ ]:
# ============================================================
# 3.3 AGGREGATE ATTENTION (200 per class)
# ============================================================
N_AGG = 200
b_ag_idx = np.random.choice(benign_idx, min(N_AGG, len(benign_idx)), replace=False)
p_ag_idx = np.random.choice(phish_idx, min(N_AGG, len(phish_idx)), replace=False)

b_cams = []
for i in tqdm(b_ag_idx, desc='Benign CAMs'):
    t = prepare_tensor(qr_val[i]); t.requires_grad_(True)
    with torch.enable_grad():
        b_cams.append(gradcam(t))
p_cams = []
for i in tqdm(p_ag_idx, desc='Phishing CAMs'):
    t = prepare_tensor(qr_val[i]); t.requires_grad_(True)
    with torch.enable_grad():
        p_cams.append(gradcam(t))

avg_b = np.mean(b_cams, axis=0)
avg_p = np.mean(p_cams, axis=0)
diff = avg_p - avg_b

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, data, title in [(axes[0], avg_b, 'Avg attention — Benign'),
                          (axes[1], avg_p, 'Avg attention — Phishing'),
                          (axes[2], diff, 'Difference (P - B)')]:
    cmap = 'RdBu_r' if 'Difference' in title else 'jet'
    vmax = abs(data).max() if 'Difference' in title else 1.0
    vmin = -vmax if 'Difference' in title else 0
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontweight='bold'); ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle(f'Class-Level Spatial Attention (n={N_AGG} each)', fontweight='bold', y=1.02)
plt.tight_layout()
save_path = os.path.join(BASE, 'fig_gradcam_aggregate.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_gradcam_aggregate.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

---
## 4. EMBEDDING SPACE ANALYSIS

In [ ]:
# ============================================================
# 4. EMBEDDING DISTANCES
# ============================================================
classifier.eval()
embeddings = []
with torch.no_grad():
    for i in range(0, len(qr_val), 128):
        batch = qr_val[i:i+128]
        tensors = torch.cat([prepare_tensor(q) for q in batch], dim=0)
        embeddings.append(classifier.backbone(tensors).cpu().numpy())
embeddings = np.concatenate(embeddings)
print(f'Embeddings: {embeddings.shape}')

SAMPLE = 500
b_emb = embeddings[lab_val == 0]
p_emb = embeddings[lab_val == 1]
if len(b_emb) > SAMPLE: b_emb = b_emb[np.random.choice(len(b_emb), SAMPLE, replace=False)]
if len(p_emb) > SAMPLE: p_emb = p_emb[np.random.choice(len(p_emb), SAMPLE, replace=False)]

bb = pdist(b_emb)
pp = pdist(p_emb)
bp = []
for i in range(min(200, len(b_emb))):
    for j in range(min(200, len(p_emb))):
        bp.append(np.linalg.norm(b_emb[i] - p_emb[j]))
bp = np.array(bp)

fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0, max(bb.max(), pp.max(), bp.max()), 60)
ax.hist(bb, bins=bins, alpha=0.5, color='#2ecc71', label=f'Benign-Benign (μ={bb.mean():.3f})', density=True)
ax.hist(pp, bins=bins, alpha=0.5, color='#e74c3c', label=f'Phish-Phish (μ={pp.mean():.3f})', density=True)
ax.hist(bp, bins=bins, alpha=0.5, color='#3498db', label=f'Benign-Phish (μ={bp.mean():.3f})', density=True)
ax.set_xlabel('Euclidean distance'); ax.set_ylabel('Density')
ax.set_title('Embedding Space Distance Distributions', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_path = os.path.join(BASE, 'fig_embedding_distances.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_embedding_distances.png', dpi=300, bbox_inches='tight')
plt.show()

intra = (bb.mean() + pp.mean()) / 2
inter = bp.mean()
ratio = inter / intra if intra > 0 else 0
print(f'\nIntra-class: {intra:.4f}')
print(f'Inter-class: {inter:.4f}')
print(f'Separation ratio (>1 = good): {ratio:.4f}')

---
## 5. SHAP ANALYSIS

In [ ]:
# ============================================================
# 5. SHAP ON EMBEDDING DIMENSIONS
# ============================================================
import shap

idx_s = np.random.choice(len(embeddings), min(500, len(embeddings)), replace=False)
X_emb = embeddings[idx_s]

def predict_from_emb(emb_np):
    with torch.no_grad():
        emb = torch.from_numpy(emb_np.astype(np.float32)).to(device)
        return torch.sigmoid(classifier.head(emb)).cpu().numpy().flatten()

background = X_emb[np.random.choice(len(X_emb), 50, replace=False)]
explainer = shap.KernelExplainer(predict_from_emb, background)
X_explain = X_emb[:100]
print('Computing SHAP (100 samples × 128 features)...')
shap_values = explainer.shap_values(X_explain, nsamples=50)

mean_shap = np.abs(shap_values).mean(axis=0)
top = np.argsort(mean_shap)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(20), mean_shap[top][::-1], color='#9b59b6', edgecolor='black')
ax.set_yticks(range(20)); ax.set_yticklabels([f'dim_{d}' for d in top[::-1]])
ax.set_xlabel('Mean |SHAP value|'); ax.set_title('Top-20 Embedding Dimensions by SHAP Importance', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_path = os.path.join(BASE, 'fig_shap_embedding.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.savefig('/content/fig_shap_embedding.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

In [ ]:
# ============================================================
# 6. SUMMARY
# ============================================================
print('='*60)
print(' XAI ANALYSIS COMPLETED')
print('='*60)
print(f'\nFigures saved to:')
print(f'  Drive: {BASE}/')
print(f'  Local backup: /content/')
print(f'\nFigures:')
for name in ['fig_gradcam_samples.png', 'fig_gradcam_aggregate.png',
              'fig_embedding_distances.png', 'fig_shap_embedding.png']:
    p = os.path.join(BASE, name)
    if os.path.exists(p):
        print(f'  OK  {name}  ({os.path.getsize(p)/1024:.0f} KB)')
print(f'\nEmbedding separation ratio: {ratio:.4f}')
print('\nAll done.')